# Практическая работа №7. «Умная корзина»: рекомендательная система на основе поведения покупателей

<a id=content></a>
### Содержание
<br>[Описание проекта](#descr_proj)
<br>[Описание данных](#descr_data)
<br>
1. [<b>Загрузка данных и получение общей информации</b>](#1)
2. [<b>Реализуйте коллаборативную фильтрацию данных на основе пользователей с применением модели `kNN NearestNeighbors`</b>](#2)
    <br>[2.1. Подготовка данных](#2.1.)
    <br>[2.2. Построение модели `kNN NearestNeighbors`](#2)
    <br>[2.3. Проверка модели `kNN NearestNeighbors` на примерах пользователей `id=4` и `id=21`](#2)
4. [<b>Вывод</b>](#3)

<a id=descr_proj></a>
### Описание проекта

<br>Постройте рекомендательную систему на примере данных о покупках. Сеть продуктовых магазинов разрабатывает новое мобильное приложение, позволяющее покупателям размещать заказы удалённо со своих мобильных устройств. Приложение должно показывать рекомендации: когда покупатель впервые нажимает на страницу «заказ», мы можем порекомендовать добавить в его корзину 10 лучших товаров, например, одноразовую посуду, свежее мясо, чипсы и т. д.
<br>Цель работы: получить список рекомендаций для указанного пользователя с помощью функции.
<br>Реализация функции для вывода рекомендаций:
* На вход — id покупателя;
* На выход — ранжированный список id товаров, которые пользователь предположительно захочет положить в свою корзину.

<a id=descr_data></a>
### Описание данных

* tgu_7_customers.csv - список из 1000 идентификаторов клиентов, рекомендуемых в качестве выходных данных;
* tgu_7_transactions.csv - информация о транзакциях клиентов.

<a id=1></a>
## Шаг 1. Загрузка данных и получение общей информации

In [1]:
# импорт библиотек
import pandas as pd
import numpy as np
from sklearn.neighbors import NearestNeighbors

import warnings
warnings.filterwarnings('ignore')

In [2]:
# импорт данных
customers = pd.read_csv('tgu_7_customers.csv')
transactions = pd.read_csv('tgu_7_transactions.csv')

In [3]:
# ознакомление с данными по покупателям
print(customers.head())
print()
print(customers.info())

   customerId
0        1553
1       20400
2       19750
3        6334
4       27773

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 1 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   customerId  1000 non-null   int64
dtypes: int64(1)
memory usage: 7.9 KB
None


In [4]:
# ознакомление с данными по транзакциям
print(transactions.head())
print()
print(transactions.info())

   customerId                        products
0           0                              20
1           1  2|2|23|68|68|111|29|86|107|152
2           2       111|107|29|11|11|11|33|23
3           3                         164|227
4           5                             2|2

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 62483 entries, 0 to 62482
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   customerId  62483 non-null  int64 
 1   products    62483 non-null  object
dtypes: int64(1), object(1)
memory usage: 976.4+ KB
None


**Выводы:**
<br>В датасете `Покупатели` мы имеем 1000 `id` клиентов. Все значения числовые. Нулевые пропущенные значения отсутствуют. Очистка данных в этом датасете не требуется.
<br>В датасете `Транзакции покупателей` мы имеем  62 483 строк с записями. `id` покупателей представлены числовыми значениями, `id` товаров - объекты. Нулевые пропущенные значения отсутствуют. Данные из этого дата сета требуют подготовки для дальнейшей работы. Необходимо разделить каждый список элементов в столбце `Продукты` на строки и подсчитать количество продуктов, приобретённых покупателем.

<a id=2></a>
## Шаг 2. Реализуйте коллаборативную фильтрацию данных на основе пользователей с применением модели `kNN NearestNeighbors`

<a id=2.1.></a>
### Шаг 2.1. Подготовка данных

In [5]:
# преобразование данных по признаку `products` в датасете `Транзакции покупателей`
transactions['products'] = transactions['products'].apply(lambda x: [int(i) for i in x.split('|')])
transactions.head(10).set_index('customerId')['products'].apply(pd.Series).reset_index()

,customerId,0,1,2,3,4,5,6,7,8,9
0,0,20.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,2.0,2.0,23.0,68.0,68.0,111.0,29.0,86.0,107.0,152.0
2,2,111.0,107.0,29.0,11.0,11.0,11.0,33.0,23.0,NaN,NaN
3,3,164.0,227.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,2.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,6,144.0,144.0,55.0,266.0,NaN,NaN,NaN,NaN,NaN,NaN
6,7,135.0,206.0,259.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,8,79.0,8.0,8.0,48.0,NaN,NaN,NaN,NaN,NaN,NaN
8,9,102.0,2.0,2.0,297.0,NaN,NaN,NaN,NaN,NaN,NaN
9,10,84.0,77.0,290.0,260.0,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
# агрегируем данные в датафрейм `data`: `id` покупателя, `id` продукта, количество покупок
data = pd.melt(transactions.set_index('customerId')['products'].apply(pd.Series).reset_index(), 
             id_vars=['customerId'],
             value_name='products') \
    .dropna().drop(['variable'], axis=1) \
    .groupby(['customerId', 'products']) \
    .agg({'products': 'count'}) \
    .rename(columns={'products': 'purchase_count'}) \
    .reset_index() \
    .rename(columns={'products': 'productId'})
data['productId'] = data['productId'].astype(np.int64)
data

,customerId,productId,purchase_count
0,0,1,2
1,0,13,1
2,0,19,3
3,0,20,1
4,0,31,2
...,...,...,...
133580,28596,211,3
133581,28596,255,1
133582,28598,212,1
133583,28604,282,1


In [7]:
# создаём матрицу
df_matrix = pd.pivot_table(data, values='purchase_count', index='customerId', columns='productId')

In [8]:
# заполняем NaN нулевыми значениями
df_matrix.fillna(0, inplace=True)

In [9]:
# выводим на ознакомление размер матрицы и первые несколько строк
print(df_matrix.shape)
df_matrix.head()

(24429, 300)


productId,0,1,2,3,4,5,6,7,8,9,...,290,291,292,293,294,295,296,297,298,299
customerId,,,,,,,,,,,,,,,,,,,,,
0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,6.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


Нами была созадна `customer id - product id` матрица, в которой строки - это пользователи, а столбцы - это товары. На пересечении строк и столбцов указаны бинарные флаги (0.0 или 1.0)

<a id=2.2.></a>
### Шаг 2.2. Построение модели `kNN NearestNeighbors`

In [10]:
# задаём параметры модели
# k=10 - это количество "похожих" пользователей
knn = NearestNeighbors(n_neighbors=11,     # добавляем 1 - это сам пользователь, при поиске ближайших соседей для пользователя он сам окажется в топе (расстояние = 0), и его нужно исключить
                       algorithm='brute',  # algorithm='brute' — точный поиск и подходит для умеренных размеров
                       metric='cosine')    # выбираем метрику: cosine — хорошо подходит для бинарных данных

# построение модели
NearestNeighbors_model = knn.fit(df_matrix)

In [11]:
# созадём функцию для составления рекомендаций топ-10 товаров для покупателя
def recommend_for_user(user_id, user_item_matrix, model, k_neighbors=10, top_n=10):
    """
    user_id: индекс пользователя в матрице (целое число)
    user_item_matrix: DataFrame или ndarray
    model: обученная NearestNeighbors
    k_neighbors: сколько похожих пользователей использовать
    top_n: сколько рекомендаций вернуть
    """
    # находим k+1 ближайших соседей (+1 потому, что мы учитываем и самого пользователя)
    distances, indices = model.kneighbors(
        user_item_matrix.iloc[[user_id]] if isinstance(user_item_matrix, pd.DataFrame) 
        else user_item_matrix[user_id:user_id+1],
        n_neighbors=k_neighbors + 1)

    # исключаем самого пользователя (первый элемент)
    similar_user_indices = indices.flatten()[1:]

    # собтираем товары, купленные похожими пользователями
    similar_users_interactions = user_item_matrix.iloc[similar_user_indices] if isinstance(user_item_matrix, pd.DataFrame) \
        else user_item_matrix[similar_user_indices]

    # суммируем "флаги", то есть сколько похожих пользователей купили каждый товар
    item_scores = similar_users_interactions.sum(axis=0)

    # исключаем товары, которые пользователь уже купил
    if isinstance(user_item_matrix, pd.DataFrame):
        user_bought = user_item_matrix.iloc[user_id]
    else:
        user_bought = user_item_matrix[user_id]

    # обнуляем уже купленные товары
    item_scores = item_scores * (1 - user_bought)

    # получаем топ-10 рекомендаций по товарам к покупке
    if isinstance(item_scores, pd.Series):
        recommended_item_indices = item_scores.nlargest(top_n).index
    else:
        recommended_item_indices = np.argsort(-item_scores)[:top_n]

    # cортируем рекомендации по возрастанию id товара
    if isinstance(recommended_item_indices, pd.Index):
        sorted_recommendations = sorted(recommended_item_indices)
    else:
        sorted_recommendations = sorted(recommended_item_indices.tolist())

    return sorted_recommendations

<a id=2.3.></a>
### Шаг 2.3. Проверка модели `kNN NearestNeighbors` на примерах пользователей `id=4` и `id=21`

In [12]:
print(f"Рекомендации для пользователя {4}:")
print(recommend_for_user(4, df_matrix, NearestNeighbors_model, 10, 10))

Рекомендации для пользователя 4:
[1, 5, 7, 9, 25, 31, 33, 40, 52, 136]


In [13]:
print(f"Рекомендации для пользователя {21}:")
print(recommend_for_user(21, df_matrix, NearestNeighbors_model, 10, 10))

Рекомендации для пользователя 21:
[0, 1, 2, 3, 4, 5, 38, 142, 179, 273]


<a id=3></a>
## 3. Вывод

<br>В условиях роста популярности цифровых сервисов розничной торговли всё большее значение приобретают персонализированные решения, способные улучшить пользовательский опыт и стимулировать повторные покупки. В рамках данного проекта разработана рекомендательная система, интегрируемая в новое мобильное приложение крупной сети продуктовых магазинов. **Цель системы — автоматически предложить пользователю релевантные товары в момент первого открытия страницы оформления заказа**, тем самым ускоряя процесс выбора и повышая вероятность покупки.  
<br>**Основная задача проекта — построить модель**, которая по идентификатору клиента (`user_id`) будет генерировать ранжированный список из 10 идентификаторов товаров (`item_id`), наиболее вероятных к добавлению в корзину при новом заказе. Для этого используется историческая информация о транзакциях, содержащая данные о том, какие товары приобретались каждым клиентом. Источником данных служат два файла: `tgu_7_transactions.csv`, содержащий записи всех совершённых покупок, и `tgu_7_customers.csv`, включающий список из 1000 клиентов, для которых необходимо сформировать рекомендации.  
<br>Проект реализует **подход коллаборативной фильтрации на основе пользовательского поведения.** Этот метод предполагает, что пользователи с похожими прошлыми покупками склонны интересоваться схожими товарами в будущем. Для поиска ближайших «соседей» — клиентов с максимально схожими предпочтениями — применяется алгоритм `kNN (k-Nearest Neighbors)`, реализованный через класс `NearestNeighbors` из библиотеки `scikit-learn`. Перед обучением модели исходные данные подвергаются трансформации: создаётся матрица «пользователи–товары», где каждая ячейка отражает факт или частоту покупки конкретного товара конкретным клиентом. Эта матрица нормализуется, чтобы нивелировать различия в общем объёме покупок между пользователями.  
<br>Построенная модель тестируется на примере двух конкретных клиентов — с идентификаторами `4` и `21`. Для каждого из них находятся ближайшие соседи, после чего на основе их покупок формируется агрегированный список наиболее популярных среди них товаров. Результирующий список ранжируется по убыванию популярности (или взвешенной оценке на основе близости соседей) и возвращается как рекомендация. Такой подход не только учитывает индивидуальные предпочтения пользователя, но и использует коллективный опыт схожих покупателей, обеспечивая баланс между персонализацией и разнообразием рекомендаций.  
<br>В итоге система способна в реальном времени генерировать актуальные, контекстно релевантные предложения для новых или редко делающих заказы пользователей, что может значительно повысить конверсию и удержание клиентов в условиях высокой конкуренции на рынке онлайн-ритейла.